In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import html
import emoji
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import pandas as pd

#only run once
# nltk.download('punkt')
# nltk.download('stop_words')
# nltk.download('wordnet')


In [2]:
df = pd.read_csv('judge-1377884607_tweet_product_company.csv',encoding='latin1')
df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
 #   Column                                              Non-Null Count  Dtype 
---  ------                                              --------------  ----- 
 0   tweet_text                                          9092 non-null   object
 1   emotion_in_tweet_is_directed_at                     3291 non-null   object
 2   is_there_an_emotion_directed_at_a_brand_or_product  9093 non-null   object
dtypes: object(3)
memory usage: 213.2+ KB


From the Non-Null Count we can see there is 1 tweet which is a null value. That probably means that the tweet is an empty string. 

I will drop this empty string

In [4]:
df.describe()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
count,9092,3291,9093
unique,9065,9,4
top,RT @mention Marissa Mayer: Google Will Connect...,iPad,No emotion toward brand or product
freq,5,946,5389


In [5]:
df['is_there_an_emotion_directed_at_a_brand_or_product'].unique()

array(['Negative emotion', 'Positive emotion',
       'No emotion toward brand or product', "I can't tell"], dtype=object)

When cleaning the text I want to do the following:
1. Remove any starting or trailing white spaces
2. Ensure every word is lower case
3. Remove any numbers
4. Remove any special characters
5. Remove any emojis (if they dont count as special characters)
6. Tokenize
7. Lemmertize
8. Remove any stopwords
9. Remove any words that repeat consecutively
10. remove any letters that repeat more than twice
11. Remove any word that is connected to the # symbol (any words connected to special symbols)
12. Remove any links they all look like this {link}
13. Remove any empty lines.
14. Remove any . symbols that repeat multiple times
15. Remove special symbols connected to numbers 5,000
16. Remove @gmail.com or @yahoo.com
17. Remove .com

In [6]:
class DropNullRows(BaseEstimator,TransformerMixin):
    #custom transformer to drop rows where the input is null because its probably a corrupt tweet or empty tweet
    def fit(self,X,y=None):
        return self
    
    def transform(self,X):
        #return the tows where the tweet is not null
        return X.dropna()



In [7]:
class TweetPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, 
                 remove_stopwords=True, 
                 min_token_len=3, 
                 lemmatize_words=True,
                 output_format='tokens'):  # 'tokens' or 'string'
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))
        self.remove_stopwords = remove_stopwords
        self.min_token_len = min_token_len
        self.lemmatize_words = lemmatize_words
        self.output_format = output_format

    def clean_text(self, text):
        if not isinstance(text,str):
            text = str(text) if text is not None else ''
        # # 1. Unescape HTML
        # text = html.unescape(text)

        # 2. Remove emojis
        text = emoji.replace_emoji(text, replace='')

        # 3. Lowercase
        text = text.lower()

        # 4. Remove retweet tag
        text = re.sub(r'^rt\s+', '', text)

        # 5. Remove URLs
        text = re.sub(r'https?://\S+', '', text)

        # 6. Remove mentions, hashtags, and cashtags
        text = re.sub(r'[@#\$]\w+', '', text)

        # 7. Remove numbers
        text = re.sub(r'\d+', '', text)

        # 8. Reduce repeated characters (e.g. soooo → so)
        text = re.sub(r'(.)\1{2,}', r'\1', text)

        # 9. Remove non-alphabetic characters
        text = re.sub(r'[^a-z\s]', '', text)

        # 10. Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        # 11. Tokenize
        tokens = word_tokenize(text)

        # 12. Filter stopwords and short tokens
        if self.remove_stopwords:
            tokens = [word for word in tokens if word not in self.stop_words]
        if self.min_token_len > 0:
            tokens = [word for word in tokens if len(word) >= self.min_token_len]

        # 13. Lemmatize
        if self.lemmatize_words:
            tokens = [self.lemmatizer.lemmatize(word) for word in tokens]

        if self.output_format == 'string':
            return ' '.join(tokens)
        return tokens

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.apply(self.clean_text)


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from xgboost import XGBClassifier

# Preprocessing pipeline (you already have this)
text_pipeline = Pipeline([
    ('preprocess', TweetPreprocessor(output_format='string')),
    ('vectorizer', TfidfVectorizer())
])

# Base models
estimators = [
    ('lr', LogisticRegression(solver='liblinear')),
    ('svc', SVC(probability=True)),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', verbosity=0))
]

# Stacking classifier
stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

# Full pipeline: text preprocessing + stacking model
pipeline = Pipeline([
    ('text', text_pipeline),
    ('model', stack_model)
])


In [11]:
df = df.dropna(subset=['tweet_text'])

X = df['tweet_text']
y = df['is_there_an_emotion_directed_at_a_brand_or_product']

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

pipeline.fit(X_train, y_train)


c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Pipeline(steps=[('text',
                 Pipeline(steps=[('preprocess',
                                  TweetPreprocessor(output_format='string')),
                                 ('vectorizer', TfidfVectorizer())])),
                ('model',
                 StackingClassifier(cv=5,
                                    estimators=[('lr',
                                                 LogisticRegression(solver='liblinear')),
                                                ('svc', SVC(probability=True)),
                                                ('xgb',
                                                 XGBClassifier(base_score=None,
                                                               booster=None,
                                                               colsample_bylevel=None,
                                                               colsample_bynode=None,
                                                               c...
                                                               max_delta_step=None,
                                                               max_depth=None,
                                                               min_child_weight=None,
                                                               missing=nan,
                                                               monotone_constraints=None,
                                                               n_estimators=100,
                                                               n_jobs=None,
                                                               num_parallel_tree=None,
                                                               random_state=None,
                                                               reg_alpha=None,
                                                               reg_lambda=None,
                                                               scale_pos_weight=None,
                                                               subsample=None,
                                                               tree_method=None,
                                                               use_label_encoder=False,
                                                               validate_parameters=None,
                                                               verbosity=0))],
                                    final_estimator=LogisticRegression(),
                                    n_jobs=-1))])

In [12]:
from sklearn.metrics import classification_report

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))


                                    precision    recall  f1-score   support

                      I can't tell       0.00      0.00      0.00        39
                  Negative emotion       0.59      0.23      0.33       151
No emotion toward brand or product       0.70      0.84      0.76      1320
                  Positive emotion       0.63      0.51      0.57       763

                          accuracy                           0.68      2273
                         macro avg       0.48      0.40      0.42      2273
                      weighted avg       0.66      0.68      0.66      2273



c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\franc\anaconda3\envs\learn-env\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
